# ITIT Analysis — Impression-to-Install Time CDF

| Field | Value |
|-------|-------|
| **Author** | Haewon |
| **Last update** | 2026-04-22 |
| **Objective** | Compare ITIT distribution by install type (VT vs CT) and publisher type (KakaoTalk vs Non-KakaoTalk) across specified bundles or campaigns |
| **Scope** | Configurable — see Parameters cell |
| **Source table** | `focal-elf-631.prod_stream_view.cv` |
| **Original analysis** | `kr_vt_deepdive.ipynb` Section 5-B |

---

**How to use:**
1. Edit the **Parameters** cell (cell below setup)
2. Run all cells top to bottom
3. Charts are saved to the same folder as this notebook

**Key metric:** `ITIT = TIMESTAMP_DIFF(cv.happened_at, imp.happened_at, SECOND)` — time from ad impression to install.
Expected pattern: CT clusters at < 5 min; VT spreads across the 24 h attribution window.

In [1]:
import os
from google.cloud import bigquery
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

In [ ]:
client = bigquery.Client(project="moloco-ods")

def run_query(query, label=''):
    df = client.query(query).result().to_dataframe()
    print(f'✅ {label}: {len(df)} rows')
    return df

CHART_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'charts'))  # VT/charts/

## Parameters

**Edit this cell to configure your analysis.** Everything else runs automatically.

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  PARAMETERS — edit this cell only
# ─────────────────────────────────────────────────────────────────

DATE_START     = '2026-01-01'   # analysis start date (YYYY-MM-DD)
DATE_END       = '2026-01-31'   # analysis end date   (YYYY-MM-DD)
TARGET_COUNTRY = 'KOR'          # 3-letter ISO country code (e.g. 'KOR', 'JPN', 'USA')
OS             = 'ANDROID'      # 'ANDROID' or 'IOS'

# ── Bundle list ──────────────────────────────────────────────────
# Option A: specify app bundles directly
#   Android: use package name e.g. 'com.netmarble.tskgb'
#   iOS:     use numeric App Store ID e.g. '6737408689'  (no 'id' prefix)
BUNDLES = [
    # 'com.kakao.games.a3korea',
    # 'com.netmarble.tskgb',
]

# Option B: specify campaign IDs — bundles will be looked up automatically
# Leave BUNDLES empty and fill CAMPAIGN_IDS to use this option
CAMPAIGN_IDS = [
    # '12345678',
    # '87654321',
]

# ── Display settings ─────────────────────────────────────────────
ITIT_CAP_HR = 25   # x-axis cap in hours (VT window = 24 h + 1 h buffer)

# ─────────────────────────────────────────────────────────────────
print(f'Period : {DATE_START} → {DATE_END}')
print(f'Country: {TARGET_COUNTRY} | OS: {OS}')
print(f'Bundles supplied: {len(BUNDLES)} | Campaign IDs supplied: {len(CAMPAIGN_IDS)}')

## Bundle Resolution

If `CAMPAIGN_IDS` was provided, look up the corresponding app bundles from `fact_dsp_creative`.
If `BUNDLES` was provided directly, this step is skipped.

In [ ]:
if BUNDLES:
    resolved_bundles = BUNDLES
    print(f'Using {len(resolved_bundles)} bundles from BUNDLES parameter.')

elif CAMPAIGN_IDS:
    campaign_ids_sql = "', '".join(CAMPAIGN_IDS)
    query_bundles = f"""
    SELECT DISTINCT product.app_market_bundle AS bundle
    FROM `moloco-ae-view.athena.fact_dsp_creative`
    WHERE date_utc >= '{DATE_START}'
      AND date_utc <= '{DATE_END}'
      AND campaign_id IN ('{campaign_ids_sql}')
      AND product.app_market_bundle IS NOT NULL
    """
    df_bundles = run_query(query_bundles, label='Bundle lookup from campaign IDs')
    resolved_bundles = df_bundles['bundle'].tolist()
    print(f'Resolved {len(resolved_bundles)} bundles: {resolved_bundles}')

else:
    raise ValueError('Set at least one of BUNDLES or CAMPAIGN_IDS in the Parameters cell.')

# Build SQL-safe IN-clause string
bundles_sql = "', '".join(resolved_bundles)

## ITIT Query

Fetches install-level impression-to-install time from `prod_stream_view.cv`.

- **VT flag:** `cv.view_through = TRUE` (canonical BOOL field; `engaged_view_through` is a separate category)
- **Publisher type:** `req.app.bundle = kakao_bundle` → KakaoTalk vs NonKakaoTalk
- **Partition key:** `timestamp` (processing time) — used for partition pruning; `happened_at` = actual event time
- Only rows with a matched impression (`imp.happened_at IS NOT NULL`) and non-negative ITIT are included

In [ ]:
query_itit = f"""
SELECT
  api.product.app.store_id                                        AS app_bundle,
  CASE WHEN cv.view_through = TRUE  THEN 'VT' ELSE 'CT' END       AS install_type,
  TIMESTAMP_DIFF(cv.happened_at, imp.happened_at, SECOND)         AS itit_sec
FROM `focal-elf-631.prod_stream_view.cv`
WHERE cv.event = 'INSTALL'
  AND timestamp BETWEEN TIMESTAMP('{DATE_START}') AND TIMESTAMP('{DATE_END}')
  AND DATE(cv.happened_at) BETWEEN '{DATE_START}' AND '{DATE_END}'
  AND api.product.app.store_id IN ('{bundles_sql}')
  AND imp.happened_at IS NOT NULL
  AND TIMESTAMP_DIFF(cv.happened_at, imp.happened_at, SECOND) >= 0
"""

df_itit = run_query(query_itit, label=f'ITIT — {OS}, {TARGET_COUNTRY}, {DATE_START}:{DATE_END}')
df_itit['segment'] = df_itit['install_type']

print('\nInstall counts by segment:')
print(df_itit.groupby('segment').size().to_string())
df_itit.head()

## ITIT CDF — All Bundles Combined

In [ ]:
SEG_ORDER  = ['VT', 'CT']
SEG_COLORS = {'VT': '#EF553B', 'CT': '#636EFA'}

fig_itit = go.Figure()
summary_rows = []

for seg in SEG_ORDER:
    d = df_itit[df_itit['segment'] == seg]['itit_sec'].dropna()
    if d.empty:
        continue
    d_hr = (d / 3600).clip(upper=ITIT_CAP_HR)
    sorted_d = np.sort(d_hr)
    cdf = np.arange(1, len(sorted_d) + 1) / len(sorted_d)
    fig_itit.add_trace(go.Scatter(
        x=sorted_d, y=cdf,
        mode='lines', name=seg,
        line=dict(color=SEG_COLORS.get(seg))
    ))
    summary_rows.append({
        'Segment':           seg,
        'N installs':        len(d),
        'Median ITIT (min)': round(d.median() / 60, 1),
        '% within 5 min':   round((d <= 300).mean() * 100, 1),
        '% within 1 hr':    round((d <= 3600).mean() * 100, 1),
        '% within 24 hr':   round((d <= 86400).mean() * 100, 1),
    })

fig_itit.update_layout(
    title=f'ITIT CDF — {OS} · {TARGET_COUNTRY} · {DATE_START} to {DATE_END}',
    xaxis_title='Hours since impression',
    yaxis_title='Cumulative % of installs',
    yaxis=dict(tickformat='.0%'),
    legend_title='Segment',
    height=450,
)
fig_itit.show()
try:
    fig_itit.write_image(
        os.path.join(CHART_DIR, f'itit_cdf_{TARGET_COUNTRY}_{OS}_{DATE_START}_{DATE_END}.png'),
        width=1000, height=450, scale=2
    )
except Exception as _e:
    print(f'⚠️  Image save skipped (install kaleido and restart runtime): {_e}')

print('\nITIT Summary:')
display(pd.DataFrame(summary_rows).set_index('Segment'))

## ITIT CDF — Per Bundle

One chart per app bundle. Useful for comparing user quality across titles.

In [ ]:
for bundle in sorted(df_itit['app_bundle'].unique()):
    df_b = df_itit[df_itit['app_bundle'] == bundle]
    fig = go.Figure()
    has_data = False

    for seg in SEG_ORDER:
        d = df_b[df_b['segment'] == seg]['itit_sec'].dropna()
        if d.empty:
            continue
        has_data = True
        d_hr = (d / 3600).clip(upper=ITIT_CAP_HR)
        sorted_d = np.sort(d_hr)
        cdf = np.arange(1, len(sorted_d) + 1) / len(sorted_d)
        fig.add_trace(go.Scatter(
            x=sorted_d, y=cdf,
            mode='lines', name=seg,
            line=dict(color=SEG_COLORS.get(seg))
        ))

    if not has_data:
        print(f'No data for {bundle}, skipping.')
        continue

    fig.update_layout(
        title=f'ITIT CDF — {bundle} ({OS}, {TARGET_COUNTRY})',
        xaxis_title='Hours since impression',
        yaxis_title='Cumulative % of installs',
        yaxis=dict(tickformat='.0%'),
        legend_title='Segment',
        height=400,
    )
    fig.show()
    safe_bundle = bundle.replace('.', '_').replace('/', '_')
    try:
        fig.write_image(
            os.path.join(CHART_DIR, f'itit_cdf_{safe_bundle}_{OS}_{DATE_START}_{DATE_END}.png'),
            width=900, height=400, scale=2
        )
    except Exception as _e:
        print(f'⚠️  Image save skipped (install kaleido and restart runtime): {_e}')